In [1]:
!pip install groq

In [2]:
from google.colab import files

uploaded = files.upload()

Saving Anees_Curated_Word_Bank.csv to Anees_Curated_Word_Bank (3).csv


In [3]:
import pandas as pd

df = pd.read_csv("Anees_Curated_Word_Bank.csv")

df.head()

,الكلمة,المعنى المرجعي,مصدر المعنى
0,السَّنا,نور,معجم الرياض
1,الشَّفَق,حمرة تظهر في الأفق من بقية ضوء الشمس بعد غروبها‏,معجم الرياض
2,الغُروب,أُفُول‏,معجم الرياض
3,الفَجر,أول وقت النهار وهو وقت ظهور ضوء خفيف,معجم الرياض
4,النَّدى,بخار المَاء يتكاثف فِي طَبَقَات الجو الْبَارِد...,معجم الرياض


In [4]:
print("Number of words:", len(df))
print("Columns:", df.columns.tolist())

Number of words: 100
Columns: ['الكلمة', 'المعنى المرجعي', 'مصدر المعنى']


In [5]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

print("Groq connected successfully!")

Groq connected successfully!


In [6]:
word = df.iloc[0]["الكلمة"]
meaning = df.iloc[0]["المعنى المرجعي"]

print("الكلمة:", word)
print("المعنى المرجعي:", meaning)

الكلمة: السَّنا
المعنى المرجعي: نور


In [7]:
for model in client.models.list().data:
    print(model.id)

groq/compound
groq/compound-mini
qwen/qwen3.6-27b
whisper-large-v3
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b


In [8]:
class AneesWordGame:

    def __init__(self, client, age_range="6 إلى 11 سنة", debug=False):
        self.client = client
        self.age_range = age_range
        self.debug = debug

    def ask_groq(self, prompt, temperature=0.3, reasoning_effort="none"):
        response = self.client.chat.completions.create(
            model="qwen/qwen3.6-27b",
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            reasoning_effort=reasoning_effort
        )
        return response.choices[0].message.content.strip()

    def generate_question(self, word, meaning, max_attempts=3):

        def normalize(text):
            text = text.replace('ال', '', 1) if text.startswith('ال') else text
            diacritics = 'ًٌٍَُِّْ'
            return ''.join(c for c in text if c not in diacritics).strip()

        clean_word = normalize(word)

        for attempt in range(max_attempts):

            prompt = f"""
أنت مساعد تعليمي للأطفال في منصة أنيس.

الفئة العمرية المستهدفة: {self.age_range}

الكلمة:
{word}

المعنى المرجعي:
{meaning}

المهمة:
حوّل المعنى المرجعي إلى سؤال قصير وواضح يستطيع الطفل معرفة الكلمة
الصحيحة من خلاله.

لا تجعل صيغة السؤال ثابتة لكل الكلمات؛ اختر صياغة تناسب نوع الكلمة
ووظيفتها ومعناها.

قواعد مهمة:

- يجب أن تكون بداية السؤال صحيحة نحويًا.
- استخدم "ما هي" إذا كان الاسم الذي يليها مؤنثًا.
- استخدم "ما هو" إذا كان الاسم الذي يليها مذكرًا.
- لا تستخدم "ما هو" قبل اسم مؤنث، ولا "ما هي" قبل اسم مذكر.
- يجب أن تكون الجملة صحيحة نحويًا بالكامل.
- استخدم عبارة أو فئة عامة تصف نوع الكلمة، وتكون مختلفة عن الكلمة الصحيحة.
- صف مشهدًا أو موقفًا ملموسًا يستطيع الطفل تخيله بسهولة.
- لا تذكر الكلمة الصحيحة أو أي جزء منها في السؤال إطلاقًا.
- لا تستخدم الكلمة الصحيحة داخل مثال أو وصف أو موقف.
- استخدم المعنى المرجعي كأساس للسؤال.
- يمكنك إضافة وصف أو موقف يساعد الطفل على فهم الكلمة، حتى لو لم يكن
  مذكورًا حرفيًا في المعنى المرجعي، بشرط ألا يغيّر المعنى الأساسي
  للكلمة أو يجعل إجابة أخرى صحيحة.
- يجب أن يحافظ السؤال على نوع الكلمة ووظيفتها كما وردت في المعنى المرجعي.
- إذا كانت الكلمة اسمًا لشيء، اجعل السؤال عن الشيء نفسه، وليس عن
  استخدامه أو أثره أو شيء مرتبط به.
- إذا كانت الكلمة تدل على مكان، اجعل السؤال عن المكان نفسه.
- إذا كانت الكلمة تدل على شعور، اجعل السؤال عن الشعور نفسه.
- إذا كانت الكلمة تدل على صوت، اجعل السؤال عن الصوت نفسه.
- إذا كانت الكلمة تدل على فعل، اجعل السؤال عن الفعل نفسه.
- إذا كانت الكلمة تدل على صفة أو ملمس، اجعل السؤال عن الصفة أو الملمس نفسه.
- لا تحوّل الكلمة إلى شيء آخر مرتبط بها.
  مثال: إذا كانت الكلمة تدل على مصدر للماء، فلا تسأل عنها باعتبارها الماء نفسه.
- قبل إخراج السؤال، راجع صيغته نحويًا وتأكد من صحة التذكير والتأنيث
  والمفرد والجمع.

أمثلة على الصياغة الصحيحة حسب نوع الكلمة:

- إذا كانت الكلمة فعلًا:
  "ما هي الحركة التي يقوم بها الطفل عندما يرتفع عن الأرض؟"

- إذا كانت الكلمة اسم مكان:
  "ما هو المكان الذي تخرج منه المياه من باطن الأرض؟"

- إذا كانت الكلمة صفة:
  "ما هي الصفة التي تصف الشيء اللين عند لمسه؟"

- إذا كانت الكلمة صوتًا:
  "ما هو الصوت الذي يصدر عند تحرك أوراق الشجر؟"

- إذا كانت الكلمة اسم شيء:
  "ما هو المصباح الذي يُعلّق أو يُحمل لإضاءة المكان؟"

مهم:
هذه الأمثلة تشرح طريقة صياغة السؤال فقط، وليست كلمات يجب استخدامها.
عند إنشاء السؤال، استخدم الكلمة والمعنى المرجعي الموجودين في الأعلى
ولا تعتمد على الأمثلة كمصدر للمعنى.

- استخدم لغة عربية فصحى بسيطة مناسبة للأطفال من عمر 6 إلى 11 سنة.
- اجعل السؤال واضحًا وله إجابة صحيحة واحدة فقط.
- لا تكتب خيارات.
- أرجع السؤال فقط.
- لا تكتب شرحًا.
- لا تستخدم <think>.
"""

            question = self.ask_groq(
                prompt,
                temperature=0.3 + attempt * 0.1
            )

            clean_question = normalize(question)

            if clean_word not in clean_question:
                return question

            if self.debug:
                print(
                    f"[تحذير] السؤال سرّب الكلمة، "
                    f"إعادة المحاولة {attempt + 1}: {question}"
                )

        fallback_prompt = f"""
أنت مساعد تعليمي للأطفال في منصة أنيس.

الفئة العمرية المستهدفة: {self.age_range}

الكلمة:
{word}

المعنى المرجعي:
{meaning}

المهمة:
اكتب سؤالًا قصيرًا يصف الكلمة بصفاتها أو وظيفتها أو مكانها مباشرة،
بحيث يستطيع الطفل معرفة الكلمة من الوصف.

الشروط:
- لا تذكر الكلمة الصحيحة أو أي جزء منها إطلاقًا.
- لا تستخدم الكلمة الصحيحة داخل وصف أو مثال.
- لا تستخدم "ما معنى كلمة".
- لا تجعل السؤال عن المعنى المرجعي نفسه.
- اجعل الوصف يؤدي إلى الكلمة الصحيحة كإجابة.
- حافظ على نوع الكلمة ووظيفتها.
- ابدأ بـ"ما هو" أو "ما هي" بما يتوافق مع الاسم الذي يلي أداة السؤال.
- تأكد أن الجملة صحيحة نحويًا.
- استخدم لغة عربية فصحى بسيطة مناسبة لعمر 6 إلى 11 سنة.
- أرجع السؤال فقط.
- لا تكتب شرحًا.
- لا تستخدم <think>.
"""

        question = self.ask_groq(
            fallback_prompt,
            temperature=0.4
        )

        clean_question = normalize(question)

        if clean_word not in clean_question:

            if self.debug:
                print(f"[أسلوب بديل] نجح: {question}")

            return question

        if self.debug:
            print(f"[أسلوب بديل] لسا مسرب: {question}")

        final_prompt = f"""
أنت مساعد تعليمي للأطفال في منصة أنيس.

الكلمة الصحيحة:
{word}

المعنى المرجعي:
{meaning}

المهمة:
اكتب سؤالًا قصيرًا يصف معنى الكلمة الصحيحة بطريقة تجعل الكلمة الصحيحة
هي الإجابة الوحيدة الممكنة.

مهم جدًا:
- يجب أن تكون إجابة السؤال هي الكلمة الصحيحة "{word}".
- لا تذكر الكلمة الصحيحة "{word}" في السؤال إطلاقًا.
- لا تذكر أي جزء منها.
- لا تذكر الكلمة الصحيحة داخل مثال أو وصف.
- لا تستخدم صيغة "ما معنى كلمة".
- لا تجعل المعنى المرجعي نفسه هو الإجابة.
- حافظ على نوع الكلمة ووظيفتها.
- ابدأ بـ"ما هو" أو "ما هي" بما يتوافق مع الاسم الذي يلي أداة السؤال.
- تأكد أن الجملة صحيحة نحويًا بالكامل.
- استخدم لغة عربية فصحى بسيطة مناسبة لطفل من عمر 6 إلى 11 سنة.
- أرجع السؤال فقط.
- لا تكتب شرحًا.
- لا تستخدم <think>.
"""

        question = self.ask_groq(
            final_prompt,
            temperature=0.3
        )

        clean_question = normalize(question)

        if self.debug:
            status = (
                "نجح"
                if clean_word not in clean_question
                else "لسا مسرب"
            )

            print(
                f"[المحاولة الأخيرة] {status}: {question}"
            )

        return question

    def generate_distractors(self, word, meaning, question):

        prompt = f"""
أنت مساعد تعليمي في لعبة الكلمات في منصة أنيس.

الفئة العمرية المستهدفة: {self.age_range}

الكلمة الصحيحة:
{word}

المعنى المرجعي:
{meaning}

السؤال:
{question}

المهمة:
أنشئ 3 كلمات مشتتة مناسبة للسؤال.

الشروط:
- كلمات عربية مناسبة للأطفال من عمر 6 إلى 11 سنة.
- يجب أن تكون الكلمات الثلاث قريبة من الكلمة الصحيحة في المجال والمعنى العام، حتى تبدو خيارات معقولة للطفل.
- يجب أن تكون مرتبطة مباشرة بالسؤال، لكن لا يمكن أن تكون إجابة صحيحة له.
- تجنب الكلمات العشوائية أو البعيدة جدًا عن موضوع السؤال.
- لا تجعل الخيارات سهلة جدًا أو مختلفة تمامًا عن بعضها.
- اجعل الخيارات الثلاثة من نفس النوع العام للكلمة الصحيحة قدر الإمكان.
- ليست مرادفات للكلمة الصحيحة.
- ليست مشتقة من الكلمة الصحيحة (نفس الجذر).
- كلمة واحدة لكل خيار.
- أرجع 3 كلمات فقط، كل كلمة في سطر مستقل.
- لا تكتب شرحًا.
- لا تستخدم <think>.
"""

        response = self.ask_groq(
            prompt,
            temperature=0.4
        )

        distractors = [
            line.strip()
            for line in response.splitlines()
            if line.strip()
        ]

        return distractors[:3]

    def verify_options(self, word, meaning, question, options):

        prompt = f"""
أنت نظام تحقق للعبة الكلمات في منصة أنيس.

الفئة العمرية المستهدفة: {self.age_range}

الكلمة الصحيحة:
{word}

المعنى المرجعي:
{meaning}

السؤال:
{question}

الخيارات:
{options}

مهمتك:
تحقق من الخيارات الأربعة بمعيارين فقط:

1. الخيار 0 هو الكلمة الصحيحة ويجب أن يكون PASS دائمًا.
2. أي خيار آخر REJECT فقط إذا كان:
   - إجابة صحيحة بديلة مقبولة لنفس السؤال، أو
   - مرادفًا حرفيًا للكلمة الصحيحة.
3. غير ذلك، اعتبره PASS حتى لو كان بعيدًا شوي عن السياق.

لا تكتب أي شرح أو تفكير. أرجع فقط 4 أسطر بهذا الشكل بالضبط،
بدون أي نص إضافي قبلها أو بعدها:

0: PASS
1: PASS أو REJECT
2: PASS أو REJECT
3: PASS أو REJECT
"""

        return self.ask_groq(
            prompt,
            temperature=0.0,
            reasoning_effort="none"
        )

    def get_rejected_indexes(self, verification_result):

        import re

        rejected_indexes = []

        pattern = re.compile(
            r'^\s*(\d+)\s*:\s*(PASS|REJECT)\s*$'
        )

        for line in verification_result.splitlines():

            match = pattern.match(line.strip())

            if match:

                index = int(match.group(1))
                status = match.group(2)

                if status == "REJECT" and index != 0:
                    rejected_indexes.append(index)

        return rejected_indexes

    def generate_replacements(
        self,
        word,
        meaning,
        question,
        options,
        rejected_indexes
    ):

        prompt = f"""
أنت مساعد تعليمي لمنصة أنيس للأطفال من عمر 6 إلى 11 سنة.

الكلمة الصحيحة:
{word}

المعنى المرجعي:
{meaning}

السؤال:
{question}

الخيارات الحالية:
{options}

أرقام الخيارات المرفوضة:
{rejected_indexes}

المهمة:
اقترح كلمات بديلة للخيارات المرفوضة.

الشروط:
- أنشئ كلمة بديلة مختلفة لكل خيار مرفوض.
- يجب أن تكون الكلمات البديلة قريبة من الكلمة الصحيحة في المجال والمعنى العام، حتى تبدو خيارات معقولة للطفل.
- يجب أن تكون مرتبطة مباشرة بالسؤال، لكن لا يمكن أن تكون إجابة صحيحة له.
- تجنب الكلمات العشوائية أو البعيدة جدًا عن موضوع السؤال.
- الكلمات البديلة يجب ألا تكون إجابة صحيحة للسؤال.
- ليست مرادفات للكلمة الصحيحة.
- ليست مشتقة من الكلمة الصحيحة.
- مناسبة لطفل من عمر 6 إلى 11 سنة.
- لا تستخدم أي كلمة موجودة أصلًا في الخيارات.
- يجب أن تكون كل كلمة مختلفة عن الأخرى.
- أرجع عددًا من الكلمات يساوي عدد الخيارات المرفوضة.
- كل كلمة في سطر مستقل.
- لا تكتب أرقامًا.
- لا تكتب شرحًا.
- لا تستخدم <think>.
"""

        response = self.ask_groq(
            prompt,
            temperature=0.4
        )

        replacements = [
            line.strip()
            for line in response.splitlines()
            if line.strip()
        ]

        return replacements

    def generate_game(self, word, meaning, max_attempts=3):

        question = self.generate_question(
            word,
            meaning
        )

        distractors = self.generate_distractors(
            word,
            meaning,
            question
        )

        options = [word] + distractors[:3]

        verification_result = self.verify_options(
            word,
            meaning,
            question,
            options
        )

        rejected_indexes = self.get_rejected_indexes(
            verification_result
        )

        attempt = 1

        if self.debug:
            print(
                f"[محاولة {attempt}] "
                f"مرفوض: {rejected_indexes}"
            )
            print(verification_result)

        while rejected_indexes and attempt < max_attempts:

            attempt += 1

            replacements = self.generate_replacements(
                word,
                meaning,
                question,
                options,
                rejected_indexes
            )

            for index, replacement in zip(
                rejected_indexes,
                replacements
            ):
                options[index] = replacement

            verification_result = self.verify_options(
                word,
                meaning,
                question,
                options
            )

            rejected_indexes = self.get_rejected_indexes(
                verification_result
            )

            if self.debug:
                print(
                    f"[محاولة {attempt}] "
                    f"مرفوض: {rejected_indexes}"
                )
                print(verification_result)

        seen = set()
        duplicate_indexes = []

        for index, option in enumerate(options):

            normalized_option = option.strip().lower()

            if normalized_option in seen:
                duplicate_indexes.append(index)
            else:
                seen.add(normalized_option)

        if duplicate_indexes:

            if self.debug:
                print(
                    f"[تكرار] خيارات مكررة: "
                    f"{duplicate_indexes}"
                )

            replacements = self.generate_replacements(
                word,
                meaning,
                question,
                options,
                duplicate_indexes
            )

            for index, replacement in zip(
                duplicate_indexes,
                replacements
            ):
                options[index] = replacement

            verification_result = self.verify_options(
                word,
                meaning,
                question,
                options
            )

            rejected_indexes = self.get_rejected_indexes(
                verification_result
            )

        import random

        correct_word = word
        shuffled_options = options[:]

        random.shuffle(shuffled_options)

        correct_index = shuffled_options.index(
            correct_word
        )

        return {
            "success": len(rejected_indexes) == 0,
            "word": word,
            "meaning": meaning,
            "question": question,
            "options": shuffled_options,
            "correct_index": correct_index,
            "verification": verification_result,
            "rejected_indexes": rejected_indexes,
            "attempts": attempt,
        }

In [9]:
game = AneesWordGame(client, debug=True)

In [10]:
distractors = game.generate_distractors(
    "نقيق",
    "صوت الضفدع",
    "ما هو الصوت الذي يصدره الضفدع؟"
)

print(distractors)

['زقزقة', 'مواءة', 'صهيل']


In [16]:
sample = df.sample(1, random_state=42)
for _, row in sample.iterrows():

    word = row["الكلمة"]
    meaning = row["المعنى المرجعي"]

    print("=" * 80)
    print(f"الكلمة: {word}")

    result = game.generate_game(word, meaning)

    print("السؤال:", result["question"])
    print("الخيارات:", result["options"])
    print("الإجابة:", result["word"])
    print("Success:", result["success"])
    print("Attempts:", result["attempts"])

الكلمة: سباق
[محاولة 1] مرفوض: []
0: PASS
1: PASS
2: PASS
3: PASS
السؤال: ما هي المنافسة التي تجري بين أشخاص أو فرق للوصول إلى خط النهاية أولاً؟
الخيارات: ['سفر', 'سباق', 'لعبة', 'درس']
الإجابة: سباق
Success: True
Attempts: 1
الكلمة: سَهْل


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3.6-27b` in organization `org_01m0ht33wte349fyh29ev9746w` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199416, Requested 821. Please try again in 1m42.384s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}